###Lakeflow Declarative Pipelines aka Delta Live Tables

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date, to_timestamp
from pyspark.sql.types import DoubleType


source_path = "/Volumes/workspace/bronze/bronzevolume/bookings/data/"
target_path = "/Volumes/workspace/silver/silvervolume/bookings/data/"
checkpoint_path = "/Volumes/workspace/silver/silvervolume/bookings/checkpoint/"


rules = "booking_id IS NOT NULL AND passenger_id IS NOT NULL"

In [0]:
df_bronze = spark.readStream.format("delta").load(source_path)


In [0]:
df_transformed = df_bronze\
                .withColumn("amount", col("amount").cast(DoubleType()))\
                .withColumn("booking_date_ts", to_timestamp(col("booking_date")))\
                .withColumn("modifiedDate", current_timestamp())\
                .withWatermark("booking_date_ts", "10 minutes")\
                .withColumn("booking_date", to_date(col("booking_date")))\
                .dropDuplicates(["booking_id", "booking_date_ts"])\
                .drop("_rescued_data")

In [0]:
df_cleaned = df_transformed.filter(rules)

query = (df_cleaned.writeStream.format("delta")\
                .outputMode("append")\
                .trigger(once=True)
                .option("checkpointLocation", checkpoint_path)\
                .start(target_path))